In [ ]:
!pip -q install transformers torch torchvision pillow matplotlib timm

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image
from google.colab import files
from transformers import Owlv2Processor, Owlv2ForObjectDetection

In [ ]:
if torch.cuda.is_available():
    print("GPU is available")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is NOT available. Running on CPU")

In [ ]:
uploaded = files.upload()

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

print("image size:", image.size)
image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
model_name = "google/owlv2-base-patch16-ensemble"

processor = Owlv2Processor.from_pretrained(model_name)
model = Owlv2ForObjectDetection.from_pretrained(model_name).to(device)

print("OWLv2 loaded.")

In [ ]:
texts = [["laptop"]]
detection_threshold = 0.40

In [ ]:
inputs = processor(
    text=texts,
    images=image,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
target_sizes = torch.tensor([image.size[::-1]]).to(device)  # (height, width)

results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    target_sizes=target_sizes,
    threshold=detection_threshold,
    text_labels=texts
)

result = results[0]
boxes = result["boxes"].cpu()
scores = result["scores"].cpu()
labels = result["text_labels"]

print("Detected objects:")
for box, score, label in zip(boxes, scores, labels):
    print(f"{label}: score={score:.3f}, box={box.tolist()}")

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.imshow(image)

for box, score, label in zip(boxes, scores, labels):
    x1, y1, x2, y2 = box.tolist()
    w, h = x2 - x1, y2 - y1

    rect = patches.Rectangle(
        (x1, y1), w, h,
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

    ax.text(
        x1,
        max(0, y1 - 5),
        f"{label}: {score:.2f}",
        color="white",
        fontsize=10,
        bbox=dict(facecolor="red", alpha=0.7)
    )

ax.set_title("OWLv2 Open-Vocabulary Detection")
ax.axis("off")
plt.show()